# Sequence Assembly and Adapter Trimming

This notebook implements primer walking assembly and adapter trimming.

## Primer Walking Assembly

This section assembles overlapping primer sequences into a contig.

In [ ]:

from Bio import SeqIO

MIN_OVERLAP = 20
MAX_MISMATCH_RATE = 0.05

def find_overlap(seq1, seq2):
    max_len = min(len(seq1), len(seq2))
    best_overlap = 0
    for i in range(MIN_OVERLAP, max_len):
        suffix = seq1[-i:]
        prefix = seq2[:i]
        mismatches = sum(1 for a, b in zip(suffix, prefix) if a != b)
        if mismatches / i <= MAX_MISMATCH_RATE:
            best_overlap = i
    return best_overlap

def merge_sequences(seq1, seq2, overlap):
    return seq1 + seq2[overlap:]

def assemble_primers(fasta_file):
    sequences = [str(rec.seq) for rec in SeqIO.parse(fasta_file, "fasta")]
    contig = sequences.pop(0)
    while sequences:
        merged = False
        for i, seq in enumerate(sequences):
            overlap = find_overlap(contig, seq)
            if overlap:
                contig = merge_sequences(contig, seq, overlap)
                sequences.pop(i)
                merged = True
                break
        if not merged:
            raise RuntimeError("Assembly stalled")
    return contig


## Adapter Trimming

Adapters are removed based on device metadata in FASTA headers.

In [ ]:

import re
from Bio import SeqIO

ADAPTERS = {
    "ONT": "AGATCGGAAGAGCACACGTCTGAACTCCAGTCA",
    "ILLUMINA": "AGATCGGAAGAGCGTCGTGTAGGGAAAGAGTGT",
    "PACBIO": "ATCTCTCTCTTTTCCTCCTCCTCCGTTGTTGTT"
}

def extract_device(header):
    match = re.search(r"device=([A-Za-z0-9]+)", header)
    return match.group(1) if match else None

def trim_adapter(sequence, adapter):
    return sequence.split(adapter)[0] if adapter in sequence else sequence

def trim_fasta(input_fasta, output_fasta):
    records = []
    for record in SeqIO.parse(input_fasta, "fasta"):
        device = extract_device(record.description)
        if device in ADAPTERS:
            record.seq = trim_adapter(str(record.seq), ADAPTERS[device])
        records.append(record)
    SeqIO.write(records, output_fasta, "fasta")


## Usage

Provide `primers.fasta` and `raw_reads.fasta` files, then run the functions above.